# 2. Train and test the model

## What you will learn

- How the model is tested year by year on years it never saw ("rolling origin"), and why one year is left out
  between training and test.
- How the rebuild's step 8 fits the logit and gradient boosting year by year, and how to read the files it writes.
- What ROC-AUC and PR-AUC mean in plain words, with ten real rows from your own table that you can check by hand.

## What you need

- Notebook 1 (or at least the idea of the panel, the label and the inputs).
- Python with the packages in `model/requirements.txt` and `jupyter`.
- The real table built once by notebook 5: run `python -m tcf_credit.rebuild data build --route free` from the
  repository folder (or `--route wrds --wrds-user YOUR_NAME`). This notebook only reads the files that command
  wrote.
- On the WRDS route your table and every output hold licensed data: keep them on your own computer and clear the
  outputs before you share the notebook.

Most cells only read files. One cell repeats two models for a single test year, so that you can check the rebuild's
result by hand.


## Set up

The first cell only sets the folders. `OUT_DIR` defaults to `<repository>/build` and `DATA_DIR` to
`<repository>/data`; the environment variables `TCF_NB_OUT_DIR` and `TCF_NB_DATA_DIR` override them. It then checks
that the rebuild's files are there. If a needed file is missing, the notebook prints one line and the later cells that
need data do nothing. It never writes into `OUT_DIR` or `DATA_DIR`, and it never deletes them.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Find the repository folder (the one that holds model/src/tcf_credit). This works from the repository folder or from
# model/notebooks.
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "model" / "src" / "tcf_credit").is_dir())
SRC = root / "model" / "src"
sys.path[:0] = [str(SRC)]
old_path = os.environ.get("PYTHONPATH")
os.environ["PYTHONPATH"] = os.pathsep.join([str(SRC)] + ([old_path] if old_path else []))

DATA_DIR = Path(os.environ.get("TCF_NB_DATA_DIR", root / "data"))
OUT_DIR = Path(os.environ.get("TCF_NB_OUT_DIR", root / "build"))
RUN_DIR = OUT_DIR / "run"
NEEDED = [OUT_DIR / "panel_v2.csv", RUN_DIR / "pooled.csv", RUN_DIR / "per_year.csv",
          RUN_DIR / "predictions.csv", RUN_DIR / "params.json"]

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
ready = all(p.exists() for p in NEEDED)
print("repository:", root)
print("data folder:", DATA_DIR)
print("output folder:", OUT_DIR)
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")


## Rolling-origin testing

A model is only useful if it works on companies and years it has **not** seen. So we never test on the data we
trained on. We also respect time: in real life you only have the past. The test therefore walks forward through
the years:

> For each test year **Y**: train on reports dated in years **up to Y-2**, and test on the reports dated in year
> **Y**. Year **Y-1** is left out on purpose.

("Origin" is the point in time where the training data stops. It rolls forward by one year each time.)

**Why leave out Y-1?** The label of a report looks 365 days ahead. A report filed in, say, November of Y-1 has a
label that is only decided by the end of year Y. When you score year Y in real life, that label is still unknown.
If we trained on Y-1, we would train on answers that were not yet known on the day. The cell below checks this on
your own table.


In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    panel = pd.read_csv(OUT_DIR / "panel_v2.csv", dtype={"cik": str})
    panel["t"] = pd.to_datetime(panel["t"])
    panel["year"] = panel["t"].dt.year
    per_year = pd.read_csv(RUN_DIR / "per_year.csv")
    predictions = pd.read_csv(RUN_DIR / "predictions.csv", dtype={"cik": str})
    predictions["t"] = pd.to_datetime(predictions["t"]).dt.strftime("%Y-%m-%d")
    window_end = panel["t"] + pd.Timedelta(days=365)          # the day each report's 12-month label window ends

    Y = int(per_year["year"].min())
    start_of_Y = pd.Timestamp(f"{Y}-01-01")
    for name, yr in [(f"reports of year Y-2 ({Y-2})", Y - 2),
                     (f"reports of year Y-1 ({Y-1})", Y - 1)]:
        m = panel["year"] == yr
        share = (window_end[m] < start_of_Y).mean()
        print(f"{name}: label known before 1 January {Y} for {share:.0%} of them")


For year Y-2 every label is settled before the test year starts, so it is safe to train on. For year Y-1 almost none
is: those labels are still open. That is the one-year gap.

This is the plan for each test year the rebuild ran, computed from your own table:


In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    df = panel[panel.has_prices == 1].copy()      # the model needs prices, so it tests only reports that have them
    rows = []
    for Y in sorted(per_year["year"].unique()):
        tr, te = df[df.year <= Y - 2], df[df.year == Y]
        rows.append({"test year": int(Y), "train on years": f"up to {Y-2}", "left out": int(Y - 1),
                     "train reports": len(tr), "train downgrades": int(tr.label.sum()),
                     "test reports": len(te), "test downgrades": int(te.label.sum())})
    display(pd.DataFrame(rows).set_index("test year"))


## Fit the two main engines with the package's own code

The model's testing code is in `model/src/tcf_credit/evaluate.py`. The rebuild's step 8 already tuned each model and
scored every test year; it saved the chosen settings in `OUT_DIR/run/params.json`. Here we use the package's functions
with those frozen settings to repeat **one** test year by hand, so that you can see the machinery:

- `Prep` prepares the inputs. Everything it learns (clipping limits, medians, scaling) comes from the **training rows
  only**. If it learned from the test rows, the test would be contaminated.
- `params.json` holds each model's setting (for example how hard the logit is held back from over-fitting, that is,
  from fitting the noise of the training years instead of the real pattern). The rebuild chose these settings on
  validation years that end **before** the first test year. They are frozen for every test year.
- `fit_predict` fits a model on the training rows and returns its probabilities.

We score each fit with **ROC-AUC** and **PR-AUC**, two quality numbers explained step by step further down in this
notebook.

The two engines:

- **Logit** (logistic regression): a weighted sum of the inputs, turned into a probability between 0 and 1. It is
  the main model.
- **Gradient boosting**: many small decision trees, each one correcting the mistakes of the ones before. It can find
  curved and interacting patterns that a logit cannot.


In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    import json

    from sklearn.metrics import average_precision_score, roc_auc_score

    from tcf_credit import evaluate as E
    from tcf_credit.features_v2 import V2_COLUMNS

    params = json.loads((RUN_DIR / "params.json").read_text())
    E.DROP[:] = params.get("drop", [])
    E.EXTRA[:] = params.get("extra", V2_COLUMNS + E.MARKET)
    E.ONLY[:] = ["logit", "boosting"]          # the two engines shown in this notebook

    # Prepare the table exactly as the rebuild's evaluate command does.
    data = pd.read_csv(OUT_DIR / "panel_v2.csv")
    data = E.fixed_order(data)                 # the same fixed row order as the rebuild
    data = data[data.has_prices == 1].copy()
    data["year"] = pd.to_datetime(data["t"]).dt.year
    data["ebit_interest"] = data["ebit_interest"].clip(-50, 100)   # a tiny interest bill explodes this ratio

    Y = int(per_year.loc[per_year["downgrades"].idxmax(), "year"])   # one test year, with many downgrades
    train, test = data[data.year <= Y - 2], data[data.year == Y]
    models = E.models(params)                  # name -> (how to build, inputs, impute, sectors)
    hand = {}
    for name, (make, cols_, impute, sectors) in models.items():
        _, p = E.fit_predict(make, cols_, impute, sectors, train, test)
        hand[name] = p
        roc = roc_auc_score(test.label, p)
        pr = average_precision_score(test.label, p)
        print(f"{name:<9} test year {Y}: ROC-AUC {roc:.3f}   PR-AUC {pr:.3f}")
    print("share of downgrades in the test year (the PR-AUC of a model with no skill):",
          round(test.label.mean(), 3))


A check that can fail: the by-hand fit above and the rebuild's `per_year.csv` must give the same test-year ROC-AUC
for the same model.


In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    for name in hand:
        from_rebuild = per_year[(per_year.model == name) & (per_year.year == Y)
                                & (per_year.rule == "calibrated_cut")].auc.iloc[0]
        by_hand = roc_auc_score(test.label, hand[name])
        assert abs(from_rebuild - by_hand) < 1e-6, (name, from_rebuild, by_hand)
        print(f"{name:<9} year {Y}: rebuild {from_rebuild:.4f}  hand {by_hand:.4f}  same")


## The rebuild's pooled table

`OUT_DIR/run/pooled.csv` puts all test years together, one line per model and flag rule. It is the rebuild's own result;
this notebook only reads it.


In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    pooled = pd.read_csv(RUN_DIR / "pooled.csv")
    show = pooled[pooled["rule"] == "calibrated_cut"] if "rule" in pooled else pooled
    columns = [c for c in ["model", "n", "downgrades", "roc_auc", "pr_auc"] if c in show]
    display(show[columns].set_index("model"))
    print("Models in the table:", ", ".join(show["model"].astype(str)))


**How to read this table.** One row per model, all test years together.

- `n`: the number of test reports. `downgrades`: how many of them were followed by an S&P downgrade within 12
  months.
- `roc_auc`, `pr_auc`: the two quality numbers (explained in the next sections).
- The model names: `logit` and `boosting` (the two engines), `logit_rating_only` (a logit that sees only the S&P
  rating state), and `avg_logit_boosting` (the plain average of the first two scores).

`pooled.csv` has **two rows per model**, and a column `rule` that names the *flag rule* behind the counts in that
row:

- `calibrated_cut`: a report is flagged if its score is above a line fixed from an **earlier** year's scores (the
  lecture's rule; notebook 3 explains it).
- `rank_top`: the riskiest 10% of each test year, by that year's own scores (a comparison only).

The two AUC columns are identical for both rules, because they use the scores and not the flags. We print the
`calibrated_cut` rows. The file has more columns: `base_rate` (share of reports that were downgraded), `brier` (the
average squared gap between the probability and what happened; lower is better), and the counts `flagged`, `TP`,
`FP`, `FN`, `TN` and rates `miss_rate`, `false_alarm_rate`, `precision`, which are the subject of notebook 3.


The table above also holds `logit_rating_only`, a logit that sees only the S&P rating state (rating, outlook,
CreditWatch, recent actions), and `avg_logit_boosting`, the plain average of the two scores. The rating-only line is
the "what you already know from the rating" benchmark. The question for the model is how much the other inputs add
on top.

On the lecture's real data the answer was: rating state alone, ROC-AUC 0.757; logit with all inputs, 0.802.


## ROC-AUC and PR-AUC in plain words

The model gives each report a score between 0 and 1. How good are the scores? Two numbers are used.

**ROC-AUC** answers: *pick one company that was downgraded and one that was not, at random. How often does the
model give the downgraded one the higher score?* A coin flip gives 0.5. A perfect model gives 1.0. (AUC means
"area under the curve"; the curve is the ROC curve. You do not need the curve to understand the number.)

**PR-AUC** answers: *walk down the list from the highest score to the lowest. Each time you reach a company that was
really downgraded, note what share of the companies you have walked past are real downgrades (the "precision").
Average those shares.* A model with no skill scores about the share of downgrades in the data (13.5% in the
lecture's data), not 0.5. This number cares about the top of the list, where the flags are, so it is the stricter
test when downgrades are rare. (PR-AUC here is computed as "average precision", the usual way.)

A set you can check by hand: ten real test rows from your own table. The next cell takes the five highest logit
scores in one test year and five more spread out through the rest, so that you can see both hits and misses.


In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    preds = predictions[(predictions["year"] == Y) & (predictions["model"] == "logit")].copy()
    names = panel[["cik", "t", "sp_name"]].copy()
    names["t"] = pd.to_datetime(names["t"]).dt.strftime("%Y-%m-%d")
    preds = preds.merge(names, on=["cik", "t"], how="left")
    preds = preds.sort_values("p", ascending=False, kind="stable").reset_index(drop=True)

    top = preds.head(5)
    rest = preds.iloc[5:]
    if len(rest) >= 5:
        positions = np.linspace(0, len(rest) - 1, 5).round().astype(int)
        picks = list(rest.index[positions])
    else:
        picks = list(rest.index)
    # If the test year has at least two downgrades, make sure two of them are in these ten real rows.
    have = int(top["label"].sum() + rest.loc[picks, "label"].sum()) if picks else int(top["label"].sum())
    want = min(2, int(preds["label"].sum()))
    need = want - have
    if need > 0:
        spare = rest[(rest["label"] == 1) & (~rest.index.isin(picks)) & (~rest.index.isin(top.index))]
        add = spare.iloc[np.linspace(0, len(spare) - 1, min(need, len(spare))).round().astype(int)]
        replace = [i for i in picks if rest.loc[i, "label"] == 0][-len(add):] if len(add) else []
        picks = [i for i in picks if i not in replace] + list(add.index)
    if len(picks) < 5:                         # a very small test year: fill up to ten rows
        spare = preds[~preds.index.isin(list(top.index) + picks)]
        picks += list(spare.index[:10 - len(top) - len(picks)])
    ten = preds.loc[list(top.index) + picks].drop_duplicates(subset=["cik", "t"]).copy()
    ten = ten.sort_values("p", ascending=False).reset_index(drop=True)
    ten = ten.rename(columns={"sp_name": "company", "p": "score", "label": "downgraded"})
    display(ten[["company", "cik", "t", "score", "downgraded"]])


In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    ten = ten.sort_values("score", ascending=False).reset_index(drop=True)

    # ROC-AUC by hand: compare every (downgraded, not downgraded) pair.
    up = ten[ten.downgraded == 1].score
    down = ten[ten.downgraded == 0].score
    wins = sum(1.0 if a > b else 0.5 if a == b else 0.0 for a in up for b in down)
    pairs = len(up) * len(down)
    print(f"pairs: {len(up)} downgraded x {len(down)} not = {pairs}; the downgraded one scores higher in {wins}")
    if pairs:
        print(f"ROC-AUC by hand = {wins}/{pairs} = {wins / pairs:.4f}   sklearn says "
              f"{roc_auc_score(ten.downgraded, ten.score):.4f}")
        assert abs(wins / pairs - roc_auc_score(ten.downgraded, ten.score)) < 1e-12
    else:
        print("No pair of one downgraded and one non-downgraded row in these ten rows.")

    # PR-AUC by hand: precision after each real downgrade, then the average.
    ten["flagged so far"] = range(1, len(ten) + 1)
    ten["downgrades so far"] = ten.downgraded.cumsum()
    ten["precision so far"] = (ten["downgrades so far"] / ten["flagged so far"]).round(3)
    hits = ten[ten.downgraded == 1]["precision so far"]
    if len(hits):
        print(f"precision at each real downgrade: {list(hits)} -> average {hits.mean():.4f}   sklearn says "
              f"{average_precision_score(ten.downgraded, ten.score):.4f}")
        assert abs(hits.mean() - average_precision_score(ten.downgraded, ten.score)) < 1e-3
    else:
        print("No downgrade in these ten rows, so the precision cannot be averaged.")
    display(ten)


Read the table from the top. At each real downgrade, the **precision so far** is the number of real downgrades so
far divided by the number of rows so far. The average of those shares is the PR-AUC by hand, and the cell checks it
against sklearn. A model with no skill would score about the share of downgrades among these ten rows. The ten rows are
only a hand-check: the pooled table above uses every test row.

Now the same two numbers for all the engines on your data, next to the no-skill PR-AUC:


In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    base_rate = pooled[pooled.rule == "calibrated_cut"].base_rate.iloc[0]
    summary = pooled[pooled.rule == "calibrated_cut"][["model", "roc_auc", "pr_auc"]].set_index("model")
    summary.loc["(no skill)"] = [0.5, base_rate]
    display(summary)


**What the lecture found on the real (licensed) data, for comparison.** Test years 2016-2024, 6,380 annual
reports, of which 862 were followed by an S&P downgrade within 12 months (13.5%).

| Result (real data) | Value |
|---|---|
| Logit: ROC-AUC / PR-AUC | 0.802 / 0.494 |
| Other single engines (not the fly models): ROC-AUC | 0.785 to 0.808 (boosting 0.808; no single engine is clearly better than the logit) |
| Averages that include the logit: ROC-AUC | 0.817 (logit-boosting) and 0.813 (all non-fly models); both clearly above the logit |
| S&P rating state alone: ROC-AUC | 0.757 |
| Fly brain models: ROC-AUC (explained in notebook 4) | 0.759 to 0.807 (real wiring had no consistent edge over shuffled wiring) |
| Flag line (top 10% of an earlier year's scores; explained in notebook 3) | caught 407 of 862 downgrades, with 8.7% false alarms |


**How the logit compared with the other models on the real data** (ROC-AUC, test years 2016-2024):

- The single flexible engines scored 0.785 to 0.808. Boosting (0.808) is slightly above the logit, but its
  interval includes zero, so no single engine is clearly better than the logit. The neural network (0.785) is
  clearly below.
- Both averages that include the logit are clearly above it: the logit-boosting average (0.817) by 0.014
  (interval 0.009 to 0.020; it is also ahead on PR-AUC, +0.019) and the average of all non-fly models (0.813)
  by 0.011.
- The logit on the S&P rating state alone scored 0.757.
- "The interval": the comparison redraws the companies many times (a bootstrap) to get a range for the difference
  between a model and the logit. Both averages that include the logit had the whole range above zero: the
  logit-boosting average's difference was +0.014 (interval 0.009 to 0.020; it was also +0.019 on PR-AUC), and the
  average of all non-fly models' was +0.011 (interval 0.003 to 0.019). Boosting's own difference was +0.005
  (interval -0.006 to +0.016, which includes zero).
- The logit was named the main model on 4 November 2026, before the extra models (XGBoost, random forest, neural
  network, boosting with direction rules, averages and the fly models) were run. Its set of inputs was extended
  later, so the logit reported here is not exactly the one named that day.


## Tidy up

This notebook only reads the files the rebuild wrote. It does not write into `OUT_DIR` or `DATA_DIR`, does not delete
them, and made no temporary folder, so there is nothing to clean up. To start fresh, run the rebuild command again
(see notebook 5) with `--from 1` or on empty folders.


## Check yourself

1. We test on year 2020. Which years are used for training, and which year is left out? Why?

<details><summary>Answer</summary>

Training uses reports dated up to 2018. 2019 is left out: the labels of 2019 reports (a 12-month look-ahead) are
not all known when the 2020 test starts, so training on them would use answers that were not available on the day.
</details>

2. A model has ROC-AUC 0.80. What does that say, in one sentence?

<details><summary>Answer</summary>

If you pick one downgraded and one non-downgraded company at random, the model gives the downgraded one the higher
score in about 80% of such pairs.
</details>

3. Only 13.5% of reports are followed by a downgrade. Why is a PR-AUC of 0.49 a good result, even though 0.49 is
below 0.5?

<details><summary>Answer</summary>

PR-AUC is not compared with 0.5. A model with no skill scores about the share of downgrades, 0.135. So 0.49 is more
than three times the no-skill level.
</details>
